In [0]:
%pip install -U mlflow scikit-learn skops
dbutils.library.restartPython()

In [0]:
import mlflow
import pandas as pd
from mlflow import MlflowClient
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "ieee_cis")
catalog = dbutils.widgets.get("catalog")
spark.sql(f"USE CATALOG {catalog}")

MODEL_NAME = f"{catalog}.ml.fraud_model"
mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

## 1. Load the champion and its threshold

In [0]:
# Look the alias up once, then use that version for both the model and the threshold,
# so moving the alias mid-run can't pair one version's model with another's threshold.
mv = client.get_model_version_by_alias(MODEL_NAME, "champion")
threshold = float(mv.tags["threshold"])

model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}/{mv.version}")   # sklearn flavour: predict_proba
features = list(model.feature_names_in_)                                   # the exact training columns

print(f"champion = v{mv.version}  threshold = {threshold:.4f}  features = {len(features)}")

## 2. Score

In [0]:
cols = list(dict.fromkeys(["TransactionID", "event_ts", "TransactionAmt", *features]))
test_pdf = spark.table("silver.test").select(*cols).toPandas()

fraud_score = model.predict_proba(test_pdf[features])[:, 1]   # column 1 = probability of fraud
flagged = (fraud_score >= threshold).astype(int)              # 0/1, same as gold.validation_predictions

preds_pdf = test_pdf[["TransactionID", "event_ts", "TransactionAmt"]].assign(
    fraud_score=fraud_score, flagged=flagged, model_version=int(mv.version))
(spark.createDataFrame(preds_pdf)
 .withColumn("scored_at", F.current_timestamp())
 .write.mode("overwrite").option("overwriteSchema", "true")
 .saveAsTable("gold.predictions"))

print(f"scored {len(preds_pdf):,} rows with v{mv.version}; flagged {flagged.sum():,} ({flagged.mean():.2%})")

## 3. Kaggle submission

In [0]:
submission = pd.DataFrame({"TransactionID": test_pdf["TransactionID"],
                           "isFraud": fraud_score})            # the probability, not 0/1
assert len(submission) == 506_691, f"Kaggle needs all 506,691 test rows, got {len(submission):,}"

path = f"/Volumes/{catalog}/bronze/raw/submission.csv"
submission.to_csv(path, index=False)
print(f"wrote {path}: download it from Catalog Explorer and upload it to Kaggle as a Late Submission")

After Kaggle returns a score, record it on the version you submitted (run once, by hand):
```python
client.set_model_version_tag(MODEL_NAME, "<version>", "kaggle_private_auc", "<score>")
```

## 4. Inspect: does the test period look like the validation period?

In [0]:
display(spark.sql("""
    SELECT flagged, COUNT(*) AS rows, ROUND(AVG(fraud_score), 3) AS avg_score
    FROM gold.predictions
    GROUP BY flagged
"""))

val_rate = (spark.table("gold.validation_predictions")
            .where(F.col("model_version") == int(mv.version))
            .agg(F.avg("flagged")).first()[0])
if val_rate is None:
    print(f"gold.validation_predictions holds another version; re-run 03_evaluate on v{mv.version} to compare")
else:
    print(f"flagged rate: test {flagged.mean():.2%} vs validation {val_rate:.2%}  "
          "(a big gap suggests the test period behaves differently, i.e. drift)")